# Itemset demo on a local Julia kernel

Open this notebook from a checkout of the repository in an IDE with a Julia kernel. The setup cell finds the checkout, and the next cell activates the project and installs its dependencies, so no clone or download of Julia is needed. Use Julia 1.13.2 to reproduce the recorded results; any 1.x from 1.12 on runs the cells.

The same cells run on Google Colab in `demo_colab.ipynb`, which differs only in setup: it clones the repository at an exact commit and installs Julia first.

The notebook verifies the environment and tests, then shows the Eclat engine: toy mining, benchmarks, runtime against minsup, transaction width, the stage profile, and the exact comparison with SPMF. The two applications run in their own notebooks, `market_basket.ipynb` and `image_patterns.ipynb`. Keep the ZIP revision with the output.

In [ ]:
# The notebook sits in notebooks/, so the checkout is its parent unless the working directory already is the checkout.
ROOT = abspath(get(ENV, "ITEMSET_ROOT", isfile("Project.toml") ? pwd() : dirname(pwd())))
isfile(joinpath(ROOT, "Project.toml")) ||
    error("Open the notebook from the repository root or from notebooks/, or set ITEMSET_ROOT to the checkout")
cd(ROOT)
println("Repository: ", ROOT)
println("Revision: ", ispath(".git") ? readchomp(`git rev-parse HEAD`) : "no .git folder; record the commit of this copy")


In [ ]:
using Pkg
v"1.12" <= VERSION < v"2" ||
    error("Use Julia 1.12 or later within 1.x; 1.13.2 gives the recorded results")
Pkg.activate(ROOT)
Pkg.instantiate()

using ItemsetMining
using Random
println("Julia: ", VERSION, " | threads: ", Threads.nthreads(), " | seed: ", DEFAULT_SEED)
println("Project: ", Base.active_project())
artifact_manifest = get(ENV, "ITEMSET_ARTIFACT_MANIFEST", "")
notebook_artifacts = Dict{String,String}()


In [ ]:
# Execute setup and tests directly inside this Julia kernel.
# Separate modules keep test definitions out of the notebook workspace.
Base.include(Module(:ItemsetSetupCheck), joinpath(ROOT, "scripts", "check_setup.jl"))
module ItemsetSetupTests
    include(joinpath(Main.ROOT, "test", "runtests.jl"))
end


In [ ]:
transactions = read_transactions(joinpath(ROOT, "data", "toy", "baskets.txt"))
minsup_count = support_threshold(0.5, length(transactions))
(; transactions, minsup_count)


## Hung — Eclat, tests, scalability, output size

`mine` with `variant=:baseline` runs the paper-based Eclat described in `docs/eclat.md`. The cell below mines the toy baskets at minsup 0.5 and prints the SPMF output, which matches `data/toy/baskets.expected.txt`. The test cell above checks the baseline against every toy fixture and a brute-force count.

In [ ]:
patterns = mine(transactions; minsup=0.5, variant=:baseline)
write_itemsets(stdout, patterns)

## Benchmarks, runtime, transaction width, stage profile

The next cell downloads the benchmark datasets, checks each file's SHA-256 against `data/benchmark/manifest.toml`, and prints the statistics computed from the files. Timing runs use `experiments/run.jl` as described in `experiments/README.md`. SPMF series need Java 25 or later.

The runtime cell prints the medians, quartiles, and ratios of the BBQ cohort and shows the report's runtime figure. The width cell regenerates the synthetic databases and shows the report's width figure. The last cell profiles the stages of the optimized CLI path (read, mine, write, and the share of each part) in about a minute. The figures come from `docs/report/figures/`, which `scripts/plot_runtime_curves.py` and `scripts/plot_width_curves.py` write, because the plots use matplotlib and this kernel is Julia.

Provenance: Julia 1.13.2 with one thread, SPMF 2.66, and `DEFAULT_SEED` (42) for every stochastic step. The runtime cohort is `bbq-all-variants-cli-5repeat-20261117T041922Z`, described in `docs/runtime-curves-2026-11-17.md` with its checksums and commands. The width experiment and the stage profile are described in `docs/bbq-width-profile-2026-11-27.md` and `docs/bbq-width-cli-2026-11-28.md`.

In [ ]:
# Prepare and verify the benchmarks in data/benchmark/.
module PrepareBenchmarks
    include(joinpath(Main.ROOT, "experiments", "prepare_benchmarks.jl"))
end
PrepareBenchmarks.main(String[])


In [ ]:
# Show a figure of the report, or name its path when the kernel cannot display images.
function show_figure(path)
    isfile(path) || return println("Figure not found: ", path)
    try
        display("image/png", read(path))
    catch
        println("Figure: ", path)
    end
end

In [ ]:
# Runtime against minsup: median and quartiles from a verified benchmark run (DAT-19).
# The default folder is the BBQ cohort in data/raw/. Set ITEMSET_RUNTIME_RUN to use another run folder
# that holds runs.csv, such as one from experiments/run.jl. The ratios are those of the report's ratio table.
module RuntimeCurves
    include(joinpath(Main.ROOT, "scripts", "runtime_curves.jl"))
end
configured = get(ENV, "ITEMSET_RUNTIME_RUN", "")
run_dir = isempty(configured) ?
          joinpath(ROOT, "data", "raw", "bbq-all-variants-cli-5repeat-20261117T041922Z") : configured
if !isfile(joinpath(run_dir, "runs.csv"))
    println("No runs.csv in ", run_dir, ". Put the BBQ run folder there or set ITEMSET_RUNTIME_RUN.")
else
    RuntimeCurves.main(["--out", joinpath(ROOT, "results", "figures", "runtime"),
        "--ratio", "Baseline/Optimized", "--ratio", "SPMF/Optimized", "--ratio", "PrePost/Optimized",
        "--ratio", "Baseline/PrePost",
        "Baseline=$run_dir:baseline", "Optimized=$run_dir:optimized", "SPMF=$run_dir:spmf",
        "PrePost=$run_dir:prepost"])
end
show_figure(joinpath(ROOT, "docs", "report", "figures", "runtime-curves.png"))

In [ ]:
# Transaction width against Eclat runtime (DAT-22): regenerate the eight synthetic databases from the
# frozen manifest and print the density, output volume, and matched support count of each (about 10 seconds).
# The timed runs are in docs/bbq-width-cli-2026-11-28.md and use experiments/configs/width-*.toml.
module WidthRun
    include(joinpath(Main.ROOT, "scripts", "width_experiment.jl"))
end
WidthRun.main(["pilot", "--out", joinpath(ROOT, "results", "figures", "width")])
show_figure(joinpath(ROOT, "docs", "report", "figures", "width-curves.png"))

In [ ]:
# Stage profile of the optimized CLI path (DAT-23): the time of the read, mine, and write stages and the share of
# each part of the search, the root layout, and the output at eleven points (about a minute). It needs the
# benchmarks that the first cell of this section prepares. The method is in the report, and the run in
# docs/bbq-width-profile-2026-11-27.md.
module StageProfileRun
    include(joinpath(Main.ROOT, "scripts", "stage_profile.jl"))
end
StageProfileRun.main(["--out", joinpath(ROOT, "results", "raw", "stage-profile"), "--repeats", "3",
                      joinpath(ROOT, "experiments", "configs", "stage-profile.toml")])

### Exact-map comparison and peak process memory

Put the pinned SPMF JAR at `tools/spmf/spmf.jar` using `tools/spmf/README.md`. Make Java 25 available on PATH or set `ITEMSET_JAVA` to its executable. The cell checks Chess at 90% minsup against the same SPMF Eclat output for both Julia variants. Expect two exact comparisons and two process peak RSS values in bytes. This single-point demo checks the setup. The full memory series uses five repeats at one medium threshold per dataset.

Run these commands from the repository root for the full experiments. Keep `runs.csv`, `environment.toml`, `references/` and the per-attempt `comparison.toml` files. Peak RSS includes startup, input, mining and output in the child process.

```sh
julia --startup-file=no --project=. experiments/run.jl experiments/configs/correctness.toml
julia --startup-file=no --project=. experiments/run.jl experiments/configs/peak-memory.toml
```

In [ ]:
using TOML, SHA
java = get(ENV, "ITEMSET_JAVA", "java")
jar = joinpath(ROOT, "tools", "spmf", "spmf.jar")
if isfile(jar) && (isfile(java) || Sys.which(java) !== nothing)
    release = TOML.parsefile(joinpath(ROOT, "tools", "spmf", "release.toml"))
    jar_sha = open(io -> bytes2hex(sha256(io)), jar)
    @assert jar_sha == release["sha256"] "SPMF JAR checksum differs from tools/spmf/release.toml"
    smoke = TOML.parsefile(joinpath(ROOT, "experiments", "configs", "correctness.toml"))
    smoke["series"] = [Dict("dataset" => "chess", "minsup" => [0.9])]
    smoke_dir = mktempdir(; cleanup=isempty(artifact_manifest))
    config = joinpath(smoke_dir, "smoke.toml")
    open(io -> TOML.print(io, smoke), config, "w")
    runner = joinpath(ROOT, "experiments", "run.jl")
    output = joinpath(smoke_dir, "run")
    run(`$(Base.julia_cmd()) --startup-file=no --project=$ROOT $runner $config --out $output --java $java`)
    println("variant | exact | peak RSS (bytes)")
    attempts = readdir(joinpath(output, "attempts"); join=true)
    @assert length(attempts) == 2
    for attempt in attempts
        comparison = TOML.parsefile(joinpath(attempt, "comparison.toml"))
        measured = TOML.parsefile(joinpath(attempt, "measure.toml"))
        @assert comparison["matches"] && measured["peak_rss_bytes"] > 0
        println(basename(attempt), " | " , comparison["matches"], " | ", measured["peak_rss_bytes"])
    end
    notebook_artifacts["spmf-smoke"] = output
    println("Raw comparison and measurement files: ", output)
else
    println("SPMF check skipped. Follow the Java and JAR setup above to run it.")
end

if !isempty(artifact_manifest)
    open(io -> TOML.print(io, notebook_artifacts), artifact_manifest, "w")
end


## Clean-run checklist

Restart the Julia kernel and run all cells. Record the Julia version, thread count, revision, dataset hashes, parameters, stdout, and failures. A local run does not prove that the hosted Colab runtime works; `demo_colab.ipynb` covers that.